# Rank UK Cities by Air Quality Using Open Data in Python

**Goal:** Produce a "best and worst air quality cities" ranking using official
data from multiple UK regulatory networks — the kind of analysis a data
journalist might run for a summer investigation.

**API keys required:** None (all UK regulatory networks are freely accessible)

**Aeolus features demonstrated:**
- `find_sites()` across multiple networks simultaneously
- `download()` with dict format for multi-network bulk retrieval
- Schema consistency when combining 5+ UK networks
- `metrics.time_average()` and `metrics.aq_stats()` at scale
- `viz.plot_distribution()` for cross-city comparison

In [ ]:
import aeolus
from aeolus import metrics, viz
from datetime import datetime

import pandas as pd
import matplotlib.pyplot as plt

## 1. Discover Urban Background Sites Across the UK

We focus on Urban Background sites for a fair comparison — these measure
the pollution that the general population breathes, without the direct
influence of nearby roads.

`find_sites()` accepts a list of sources and returns combined metadata
with a consistent schema.

In [ ]:
# Search across all free UK regulatory networks
uk_networks = ["AURN", "AQE", "SAQN", "WAQN", "NI"]
all_sites = aeolus.find_sites(uk_networks)

print(f"Total sites across {len(uk_networks)} networks: {len(all_sites)}")
print(f"\nSites per network:")
print(all_sites["source_network"].value_counts())

In [ ]:
# Filter to Urban Background sites
background = all_sites[
    all_sites["location_type"].str.contains("Urban Background", na=False)
].copy()

print(f"Urban Background sites: {len(background)}")
print(f"\nSample sites:")
background[["site_code", "site_name", "source_network"]].head(10)

## 2. Download Summer Data

Download NO\u2082 and PM\u2082.\u2085 for the summer months (June–August).
We use the dict format to download from multiple networks in one call.

**Note:** For a large number of sites this may take a few minutes.
Aeolus shows progress indicators when `tqdm` is installed.

In [ ]:
# Build download dict: {network: [sites]}
download_map = (
    background
    .groupby("source_network")["site_code"]
    .apply(list)
    .to_dict()
)

# Show what we're downloading
for network, sites in download_map.items():
    print(f"{network}: {len(sites)} sites")

In [ ]:
# Download summer 2024 data
# Tip: for faster iteration, start with a subset:
#   download_map = {"AURN": download_map["AURN"][:10]}
data = aeolus.download(
    download_map,
    start_date=datetime(2024, 6, 1),
    end_date=datetime(2024, 8, 31),
)

print(f"Downloaded {len(data):,} rows")
print(f"Sites: {data['site_code'].nunique()}")
print(f"Pollutants: {data['measurand'].unique().tolist()}")

## 3. Calculate Site-Level Summer Means

Use `time_average()` to compute the summer mean for each site,
then filter to sites with adequate data capture.

In [ ]:
# Focus on NO2 and PM2.5
pollutants = ["NO2", "PM2.5"]
subset = data[data["measurand"].isin(pollutants)]

# Site-level summer means
# Use a long averaging period to get one value per site
site_means = (
    subset
    .groupby(["site_code", "source_network", "measurand"])["value"]
    .agg(["mean", "count"])
    .reset_index()
)

# Require at least 1000 hourly readings (~46 days of data)
site_means = site_means[site_means["count"] >= 1000]
print(f"Sites with sufficient data: {site_means['site_code'].nunique()}")

## 4. Map Sites to Cities

We extract a city/region name from the site metadata. AURN site names
typically follow the pattern "City Sitename" (e.g., "London Marylebone Road").

In [ ]:
# Build a site-to-city lookup from metadata
# Use site_name first word as a rough city proxy
city_map = dict(zip(
    background["site_code"],
    background["site_name"].str.split().str[0],
))

site_means["city"] = site_means["site_code"].map(city_map)

# City-level means (average across sites in each city)
city_means = (
    site_means
    .groupby(["city", "measurand"])["mean"]
    .mean()
    .reset_index()
)

print(f"Cities represented: {city_means['city'].nunique()}")

## 5. Rankings

In [ ]:
for pollutant in pollutants:
    poll_data = city_means[city_means["measurand"] == pollutant].sort_values("mean", ascending=False)
    
    if poll_data.empty:
        print(f"\nNo {pollutant} data available")
        continue
    
    print(f"\n{'='*50}")
    print(f"{pollutant} \u2014 Summer 2024 Rankings")
    print(f"{'='*50}")
    
    print(f"\nTop 10 (highest):")
    for i, (_, row) in enumerate(poll_data.head(10).iterrows(), 1):
        print(f"  {i:2d}. {row['city']:<20s} {row['mean']:.1f} \u00b5g/m\u00b3")
    
    print(f"\nBottom 5 (cleanest):")
    for i, (_, row) in enumerate(poll_data.tail(5).iloc[::-1].iterrows(), 1):
        print(f"  {i:2d}. {row['city']:<20s} {row['mean']:.1f} \u00b5g/m\u00b3")

In [ ]:
# Bar chart of top/bottom cities for NO2
no2_ranking = (
    city_means[city_means["measurand"] == "NO2"]
    .sort_values("mean", ascending=True)
)

if len(no2_ranking) > 0:
    # Show top 15 and bottom 5
    n_show = min(20, len(no2_ranking))
    to_plot = pd.concat([no2_ranking.tail(15), no2_ranking.head(5)]).drop_duplicates()
    to_plot = to_plot.sort_values("mean")
    
    fig, ax = plt.subplots(figsize=(10, max(6, n_show * 0.4)))
    colours = ["#e74c3c" if v > 40 else "#f39c12" if v > 20 else "#27ae60"
               for v in to_plot["mean"]]
    
    ax.barh(to_plot["city"], to_plot["mean"], color=colours)
    ax.axvline(40, color="black", linestyle="--", linewidth=0.8, label="UK limit")
    ax.set_xlabel("Summer mean NO\u2082 (\u00b5g/m\u00b3)")
    ax.set_title("UK Cities Ranked by Summer NO\u2082 (2024)")
    ax.legend()
    plt.tight_layout()
    plt.show()

## 6. Distribution Comparison

Use `viz.plot_distribution()` to compare hourly concentration distributions
across the top cities.

In [ ]:
# Get top 6 cities by NO2 for a focused comparison
top_cities = (
    city_means[city_means["measurand"] == "NO2"]
    .nlargest(6, "mean")["city"]
    .tolist()
)

# Get site codes for these cities
top_site_codes = [
    code for code, city in city_map.items()
    if city in top_cities
]

top_data = subset[subset["site_code"].isin(top_site_codes)]

if not top_data.empty:
    fig = viz.plot_distribution(
        top_data,
        pollutant="NO2",
        group_by="site",
        style="box",
        title="NO\u2082 Distribution: Highest UK Cities (Summer 2024)",
    )
    plt.show()

## Summary

This notebook demonstrated:

1. **Multi-network site discovery** — querying 5 UK networks in one call
2. **Schema consistency** — data from AURN, AQE, SAQN, WAQN, NI combined seamlessly
3. **Bulk download** — dict format for structured multi-network retrieval
4. **Composable analysis** — `time_average()` and pandas groupby for city-level aggregation
5. **Rankings and distributions** — ready for publication

### Caveats for publication
- City assignment from site names is approximate; a proper analysis would use
  Local Authority boundaries or urban area definitions
- Summer means differ from annual means used in regulatory assessments
- Some cities may have only one monitor — not fully representative

### Next steps
- Add a geographic map using `geopandas` and site coordinates
- Extend to a full year for annual regulatory statistics
- Include trend analysis to show which cities are improving fastest